# 06 · NGSIM stop-and-go — Speed

Two-shot decomposition of follower **speed** for the NGSIM stop-and-go
trajectories. Because speed is physically non-negative, the reconstruction is
clipped at zero and the personalized component is re-derived from the clipped
reconstruction (the "corrected" component). Clustering uses the top-5 PCA scores
of the corrected personalized components with k-means (k = 3).

**Figures produced:** `NGSIM_speed_component.png`, `NGSIM_speed_cluster.png`,
`NGSIM_cluster_speed.png`.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "bhd").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
import seaborn as sns
from scipy.interpolate import interp1d
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from bhd import DATA_DIR, FIG_DIR, set_style, load_ngsim_stop_to_go

set_style(seaborn_white=True)

## 1. Load and align trajectories

In [ ]:
df = load_ngsim_stop_to_go(DATA_DIR / "ngsim" / "stop_to_go.csv")
traj_ids = sorted(df["trajectory_id"].unique())
min_len = min(len(df[df["trajectory_id"] == tid]) for tid in traj_ids)
print(f"{len(traj_ids)} trajectories, resampled to {min_len} points")

speed_list = []
for tid in traj_ids:
    speed_vals = df[df["trajectory_id"] == tid]["follower_speed"].values
    f = interp1d(np.linspace(0, 1, len(speed_vals)), speed_vals)
    speed_list.append(f(np.linspace(0, 1, min_len)))
V = np.stack(speed_list, axis=1)  # (time, vehicles)

## 2. Two-shot decomposition with positivity correction

In [ ]:
pca_shared = PCA(n_components=1)
C_speed = pca_shared.fit(V.T).components_.T
Psi_speed = C_speed.T @ V
V_shared = C_speed @ Psi_speed

personalized_speed, reconstructed_speed = [], []
for v in range(V.shape[1]):
    R_v = (V[:, v] - V_shared[:, v]).reshape(-1, 1)
    pca_personal = PCA(n_components=1)
    Phi_v = pca_personal.fit_transform(R_v)
    R_hat = pca_personal.inverse_transform(Phi_v)
    personalized_speed.append(R_hat.flatten())
    reconstructed_speed.append(V_shared[:, v] + R_hat.flatten())
V_personal = np.column_stack(personalized_speed)
V_reconstructed = np.column_stack(reconstructed_speed)
V_reconstructed_clipped = np.clip(V_reconstructed, a_min=0, a_max=None)
V_personal_corrected = np.clip(V_reconstructed_clipped - V_shared, a_min=0, a_max=None)

colors = cm.viridis(np.linspace(0, 1, V.shape[1]))
xticks = list(range(0, 401, 50))
xtick_labels = [str(x // 10) for x in xticks]

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
for i in range(V.shape[1]):
    axes[0].plot(V[:, i], color=colors[i], alpha=0.4)
    axes[1].plot(V_shared[:, i], color=colors[i], alpha=0.6)
    axes[2].plot(V_personal_corrected[:, i], linestyle="--", color=colors[i], alpha=0.8)
axes[0].set_title("True Speed", fontsize=22)
axes[1].set_title("Corrected Common Component (CΨ)", fontsize=22)
axes[2].set_title("Personalized Component (PΦ)", fontsize=22)
axes[2].set_xlabel("Time (sec)", fontsize=22)
for ax in axes:
    ax.tick_params(axis="both", labelsize=20)
    ax.set_ylim(-1, 20)
    ax.set_xlim(0, 400)
    ax.set_yticks([0, 5, 10, 15, 20])
    ax.set_ylabel("Speed (m/s)", fontsize=20)
    ax.set_xticks(xticks)
    ax.set_xticklabels(xtick_labels)
    ax.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "NGSIM_speed_component.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Clustering the personalized components

In [ ]:
pca_cluster = PCA(n_components=5)
X_cluster = pca_cluster.fit_transform(V_personal_corrected.T)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_cluster)
print("Cluster sizes:", np.bincount(labels))

plt.figure(figsize=(10, 7))
palette = sns.color_palette("Set2", n_colors=3)
for cid in range(3):
    idx = np.where(labels == cid)[0]
    plt.plot(V_personal_corrected[:, idx].mean(axis=1), label=f"Cluster {cid}", color=palette[cid], linewidth=3)
plt.xlabel("Time (sec)", fontsize=22)
plt.xticks(ticks=xticks, labels=xtick_labels, fontsize=20)
plt.xlim(0, 400)
plt.ylabel("Speed (m/s)", fontsize=22)
plt.yticks([0, 3, 6, 9, 12, 15], fontsize=20)
plt.ylim(-2.5, 15)
plt.legend(fontsize=22, loc="lower center", ncol=3)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "NGSIM_speed_cluster.png", dpi=300, bbox_inches="tight")
plt.show()

speed_2d = PCA(n_components=2).fit_transform(V_personal_corrected.T)
plt.figure(figsize=(10, 10))
sns.scatterplot(x=speed_2d[:, 0], y=speed_2d[:, 1], hue=labels, palette="Set2", s=120, edgecolor="black")
for i in range(len(labels)):
    plt.text(speed_2d[i, 0] + 0.02, speed_2d[i, 1], f"V{i + 1}", fontsize=20)
handles, legend_labels = plt.gca().get_legend_handles_labels()
new_labels = [f"Cluster {int(lbl)}" if lbl.isdigit() else lbl for lbl in legend_labels]
plt.xlabel("Personalized Component 1", fontsize=22)
plt.xticks(fontsize=20)
plt.ylabel("Personalized Component 2", fontsize=22)
plt.yticks(fontsize=20)
plt.legend(handles, new_labels, fontsize=22)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "NGSIM_cluster_speed.png", dpi=300, bbox_inches="tight")
plt.show()